In [1]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
import json

pd.set_option('display.max_columns', None)


In this step, I am importing the libraries that I need for the data work.  

* pandas – to work with tabular data  
* geopandas – to handle geographic data  
* osmnx– to retrieve pet store locations from OpenStreetMap  
* json– because some OSM data can come in JSON format  

I also used pd.set_option('display.max_columns', None) so that Jupyter shows all columns fully, without hiding or shortening them.



In [2]:
tags = {"shop": "pet"}
petstores_gdf = ox.features_from_place("Berlin, Germany", tags=tags)
petstores_gdf.head()


geometry           brand brand:wikidata  \
element id                                                                    
node    111810614  POINT (13.28841 52.49974)       Fressnapf        Q875796   
        346138343  POINT (13.36752 52.54252)       Fressnapf        Q875796   
        417360251  POINT (13.45196 52.53361)       Fressnapf        Q875796   
        438385039  POINT (13.30844 52.47888)             NaN            NaN   
        446899194  POINT (13.44802 52.46867)  Das Futterhaus       Q1167914   

                     brand:wikipedia  check_date check_date:opening_hours  \
element id                                                                  
node    111810614       en:Fressnapf  2025-10-29               2025-10-29   
        346138343       de:Fressnapf  2025-08-28               2025-08-28   
        417360251       en:Fressnapf         NaN                      NaN   
        438385039                NaN         NaN                      NaN   
        446899194  de:Das Futterhaus         NaN                      NaN   

                             name                      opening_hours shop  \
element id                                                                  
node    111810614       Fressnapf  Mo-Fr 09:00-20:00; Sa 09:00-19:00  pet   
        346138343       Fressnapf  Mo-Fr 09:00-20:00; Sa 09:00-18:00  pet   
        417360251       Fressnapf                                NaN  pet   
        438385039  Lucas Tierwelt                  Mo-Sa 08:00-20:00  pet   
        446899194  Das Futterhaus                                NaN  pet   

                  addr:housenumber        addr:street level wheelchair  \
element id                                                               
node    111810614              NaN                NaN   NaN        NaN   
        346138343            10-11       Müllerstraße     0        yes   
        417360251             128b   Storkower Straße   NaN    limited   
        438385039               95  Forckenbeckstraße   NaN        NaN   
        446899194              NaN                NaN   NaN        NaN   

                  addr:city addr:country addr:postcode      addr:suburb  \
element id                                                                
node    111810614       NaN          NaN           NaN              NaN   
        346138343       NaN          NaN           NaN              NaN   
        417360251    Berlin           DE         10407  Prenzlauer Berg   
        438385039    Berlin           DE         14199    Schmargendorf   
        446899194       NaN          NaN           NaN              NaN   

                  toilets:wheelchair  \
element id                             
node    111810614                NaN   
        346138343                NaN   
        417360251                 no   
        438385039                NaN   
        446899194                NaN   

                                              wheelchair:description  \
element id                                                             
node    111810614                                                NaN   
        346138343                                                NaN   
        417360251  Behindertenparkplätze werden dauerhaft als Abs...   
        438385039                                                NaN   
        446899194                                                NaN   

                                          website  pet phone email  \
element id                                                           
node    111810614                             NaN  NaN   NaN   NaN   
        346138343                             NaN  NaN   NaN   NaN   
        417360251                             NaN  NaN   NaN   NaN   
        438385039  https://www.lucas-tierwelt.de/  NaN   NaN   NaN   
        446899194                             NaN  NaN   NaN   NaN   

                  payment:american_express payment:coins payment:mastercard  \
e

Here, I am retrieving pet shop locations in Berlin from OpenStreetMap.

* tags = {"shop": "pet"} selects only pet store locations  
* ox.features_from_place("Berlin, Germany", tags=tags) gets pet shops located in Berlin
* "Berlin, Germany" the place boundary used for the OSM query. 
* petstores_gdf.head() displays the first few rows to verify the data loaded correctly


In [3]:

petstores_gdf["district_id"] = 3


if "index_right" in petstores_gdf.columns:
    petstores_gdf = petstores_gdf.drop(columns=["index_right"])

neighborhoods = gpd.read_file("../../districts/sources/neighborhoods_enhanced.geojson")

petstores_gdf = gpd.sjoin(
    petstores_gdf,
    neighborhoods[["district_id", "geometry"]],
    how="left",
    predicate="within",
    lsuffix="left",
    rsuffix="right"
)


petstores_gdf = petstores_gdf.rename(columns={"district_id_right": "neighborhood_id"})


petstores_gdf.head()





geometry           brand brand:wikidata  \
element id                                                                    
node    111810614  POINT (13.28841 52.49974)       Fressnapf        Q875796   
        346138343  POINT (13.36752 52.54252)       Fressnapf        Q875796   
        417360251  POINT (13.45196 52.53361)       Fressnapf        Q875796   
        438385039  POINT (13.30844 52.47888)             NaN            NaN   
        446899194  POINT (13.44802 52.46867)  Das Futterhaus       Q1167914   

                     brand:wikipedia  check_date check_date:opening_hours  \
element id                                                                  
node    111810614       en:Fressnapf  2025-10-29               2025-10-29   
        346138343       de:Fressnapf  2025-08-28               2025-08-28   
        417360251       en:Fressnapf         NaN                      NaN   
        438385039                NaN         NaN                      NaN   
        446899194  de:Das Futterhaus         NaN                      NaN   

                             name                      opening_hours shop  \
element id                                                                  
node    111810614       Fressnapf  Mo-Fr 09:00-20:00; Sa 09:00-19:00  pet   
        346138343       Fressnapf  Mo-Fr 09:00-20:00; Sa 09:00-18:00  pet   
        417360251       Fressnapf                                NaN  pet   
        438385039  Lucas Tierwelt                  Mo-Sa 08:00-20:00  pet   
        446899194  Das Futterhaus                                NaN  pet   

                  addr:housenumber        addr:street level wheelchair  \
element id                                                               
node    111810614              NaN                NaN   NaN        NaN   
        346138343            10-11       Müllerstraße     0        yes   
        417360251             128b   Storkower Straße   NaN    limited   
        438385039               95  Forckenbeckstraße   NaN        NaN   
        446899194              NaN                NaN   NaN        NaN   

                  addr:city addr:country addr:postcode      addr:suburb  \
element id                                                                
node    111810614       NaN          NaN           NaN              NaN   
        346138343       NaN          NaN           NaN              NaN   
        417360251    Berlin           DE         10407  Prenzlauer Berg   
        438385039    Berlin           DE         14199    Schmargendorf   
        446899194       NaN          NaN           NaN              NaN   

                  toilets:wheelchair  \
element id                             
node    111810614                NaN   
        346138343                NaN   
        417360251                 no   
        438385039                NaN   
        446899194                NaN   

                                              wheelchair:description  \
element id                                                             
node    111810614                                                NaN   
        346138343                                                NaN   
        417360251  Behindertenparkplätze werden dauerhaft als Abs...   
        438385039                                                NaN   
        446899194                                                NaN   

                                          website  pet phone email  \
element id                                                           
node    111810614                             NaN  NaN   NaN   NaN   
        346138343                             NaN  NaN   NaN   NaN   
        417360251                             NaN  NaN   NaN   NaN   
        438385039  https://www.lucas-tierwelt.de/  NaN   NaN   NaN   
        446899194                             NaN  NaN   NaN   NaN   

                  payment:american_express payment:coins payment:mastercard  \
e

Added district_id using Spatial Join
In this step, I used the districts_enhanced.geojson file to join district information with pet store locations based on spatial relationships.



In [4]:
print("Current CRS:", petstores_gdf.crs)


Current CRS: epsg:4326


In [5]:
# ✅ Validate and clean geospatial data

print("Current CRS:", petstores_gdf.crs)


if petstores_gdf.crs != "EPSG:4326":
    petstores_gdf = petstores_gdf.to_crs(epsg=4326)
    print("CRS converted to EPSG:4326 (WGS 84)")

invalid_geometries = petstores_gdf[~petstores_gdf.is_valid]
print(f"Invalid geometries found: {len(invalid_geometries)}")

petstores_gdf = petstores_gdf.drop_duplicates(subset='geometry')

print("Geospatial validation completed successfully.")


Current CRS: epsg:4326
Invalid geometries found: 0
Geospatial validation completed successfully.


Validate and Clean Geospatial Data
* I confirmed that all geometries are valid and the coordinate reference system is EPSG:4326 (WGS 84), which is the standard for Berlin data.
No invalid geometries or duplicates were found, ensuring that the dataset is spatially accurate and ready for further cleaning and transformation.

In [6]:
petstores_gdf.columns.tolist()


['geometry',
 'brand',
 'brand:wikidata',
 'brand:wikipedia',
 'check_date',
 'check_date:opening_hours',
 'name',
 'opening_hours',
 'shop',
 'addr:housenumber',
 'addr:street',
 'level',
 'wheelchair',
 'addr:city',
 'addr:country',
 'addr:postcode',
 'addr:suburb',
 'toilets:wheelchair',
 'wheelchair:description',
 'website',
 'pet',
 'phone',
 'email',
 'payment:american_express',
 'payment:coins',
 'payment:mastercard',
 'payment:notes',
 'payment:visa',
 'contact:email',
 'contact:fax',
 'contact:phone',
 'contact:website',
 'note',
 'operator',
 'fax',
 'payment:credit_cards',
 'payment:debit_cards',
 'post_office',
 'post_office:brand',
 'post_office:brand:wikidata',
 'payment:girocard',
 'source',
 'contact:facebook',
 'delivery',
 'description',
 'species:de',
 'addr:inclusion',
 'contact:instagram',
 'ref:vatin',
 'dispensing',
 'opening_hours:covid19',
 'old_name',
 'wikimedia_commons',
 'surveillance',
 'start_date',
 'opening_hours:signed',
 'organic',
 'post_office:ref',

This lists all column names in the dataset to check which attributes are available and what information each represents.


In [7]:
petstores_gdf.isna().sum().sort_values(ascending=False)


room                          95
old_name                      95
post_office:brand:wikidata    95
payment:girocard              95
contact:fax                   95
                              ..
index_right                    0
geometry                       0
shop                           0
name                           0
neighborhood_id                0
Length: 73, dtype: int64

Here I am checking how many missing values each column has.

* .isna() marks missing entries
* .sum() counts how many missing values exist  
* .sort_values() sorts them from most to least  

This helps me understand data quality and decide which columns are useful.


In [8]:
keep_cols = [
    'geometry',
    'name',
    'brand',
    'opening_hours',
    'addr:street',
    'addr:housenumber',
    'addr:postcode',
    'addr:city',
    'phone',
    'website'
]

petstores_cleaned = petstores_gdf[keep_cols].copy()
petstores_cleaned.head()


geometry            name           brand  \
element id                                                                     
node    111810614  POINT (13.28841 52.49974)       Fressnapf       Fressnapf   
        346138343  POINT (13.36752 52.54252)       Fressnapf       Fressnapf   
        417360251  POINT (13.45196 52.53361)       Fressnapf       Fressnapf   
        438385039  POINT (13.30844 52.47888)  Lucas Tierwelt             NaN   
        446899194  POINT (13.44802 52.46867)  Das Futterhaus  Das Futterhaus   

                                       opening_hours        addr:street  \
element id                                                                
node    111810614  Mo-Fr 09:00-20:00; Sa 09:00-19:00                NaN   
        346138343  Mo-Fr 09:00-20:00; Sa 09:00-18:00       Müllerstraße   
        417360251                                NaN   Storkower Straße   
        438385039                  Mo-Sa 08:00-20:00  Forckenbeckstraße   
        446899194                                NaN                NaN   

                  addr:housenumber addr:postcode addr:city phone  \
element id                                                         
node    111810614              NaN           NaN       NaN   NaN   
        346138343            10-11           NaN       NaN   NaN   
        417360251             128b         10407    Berlin   NaN   
        438385039               95         14199    Berlin   NaN   
        446899194              NaN           NaN       NaN   NaN   

                                          website  
element id                                         
node    111810614                             NaN  
        346138343                             NaN  
        417360251                             NaN  
        438385039  https://www.lucas-tierwelt.de/  
        446899194                             NaN

In this step, I manually selected the columns that are actually useful for the project.  
Before choosing, I checked all columns and their missing value counts.  
Some columns were almost completely empty
so they did not add meaningful value to the dataset.  






In [9]:
petstores_cleaned['full_address'] = (
    petstores_cleaned['addr:street'].fillna('') + ' ' +
    petstores_cleaned['addr:housenumber'].fillna('') + ', ' +
    petstores_cleaned['addr:postcode'].fillna('') + ' ' +
    petstores_cleaned['addr:city'].fillna('')
).str.strip()

petstores_cleaned[['name', 'full_address']].head()


name                         full_address
element id                                                            
node    111810614       Fressnapf                                    ,
        346138343       Fressnapf                  Müllerstraße 10-11,
        417360251       Fressnapf  Storkower Straße 128b, 10407 Berlin
        438385039  Lucas Tierwelt   Forckenbeckstraße 95, 14199 Berlin
        446899194  Das Futterhaus                                    ,

Here I am creating a single full_address field by combining the individual address components  
(street, house number, postcode, city).

Since OSM provides these parts separately, I merge them into a clean and readable address  
format. fillna() is used to avoid issues when some fields are missing, and str.strip() removes extra spaces

Therefore, I preview the result with head() to make sure the address looks correct.




In [10]:
petstores_cleaned['geometry'] = petstores_cleaned['geometry'].centroid
petstores_cleaned['longitude'] = petstores_cleaned.geometry.x
petstores_cleaned['latitude'] = petstores_cleaned.geometry.y

petstores_cleaned[['name', 'full_address', 'latitude', 'longitude']].head()


/var/folders/mn/w43_rvg92n9dv58q76z80hgc0000gn/T/ipykernel_38402/2468793336.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  petstores_cleaned['geometry'] = petstores_cleaned['geometry'].centroid


name                         full_address  \
element id                                                               
node    111810614       Fressnapf                                    ,   
        346138343       Fressnapf                  Müllerstraße 10-11,   
        417360251       Fressnapf  Storkower Straße 128b, 10407 Berlin   
        438385039  Lucas Tierwelt   Forckenbeckstraße 95, 14199 Berlin   
        446899194  Das Futterhaus                                    ,   

                    latitude  longitude  
element id                               
node    111810614  52.499735  13.288411  
        346138343  52.542519  13.367520  
        417360251  52.533614  13.451963  
        438385039  52.478876  13.308440  
        446899194  52.468671  13.448016

The geometry field in OSM can sometimes be stored as shapes (polygons).  
However, for mapping and database storage we need a single coordinate point.  

* geometry.centroid extracts the center point of the shape.  
* geometry.x gives the longitude.  
* geometry.y gives the latitude.  

Then I display a preview to verify that the final coordinates were extracted correctly.



In [11]:
petstores_cleaned = petstores_cleaned.drop(columns=[
    'addr:street', 'addr:housenumber', 'addr:postcode', 'addr:city'
], errors='ignore')

petstores_cleaned.head()



geometry            name           brand  \
element id                                                                     
node    111810614  POINT (13.28841 52.49974)       Fressnapf       Fressnapf   
        346138343  POINT (13.36752 52.54252)       Fressnapf       Fressnapf   
        417360251  POINT (13.45196 52.53361)       Fressnapf       Fressnapf   
        438385039  POINT (13.30844 52.47888)  Lucas Tierwelt             NaN   
        446899194  POINT (13.44802 52.46867)  Das Futterhaus  Das Futterhaus   

                                       opening_hours phone  \
element id                                                   
node    111810614  Mo-Fr 09:00-20:00; Sa 09:00-19:00   NaN   
        346138343  Mo-Fr 09:00-20:00; Sa 09:00-18:00   NaN   
        417360251                                NaN   NaN   
        438385039                  Mo-Sa 08:00-20:00   NaN   
        446899194                                NaN   NaN   

                                          website  \
element id                                          
node    111810614                             NaN   
        346138343                             NaN   
        417360251                             NaN   
        438385039  https://www.lucas-tierwelt.de/   
        446899194                             NaN   

                                          full_address  longitude   latitude  
element id                                                                    
node    111810614                                    ,  13.288411  52.499735  
        346138343                  Müllerstraße 10-11,  13.367520  52.542519  
        417360251  Storkower Straße 128b, 10407 Berlin  13.451963  52.533614  
        438385039   Forckenbeckstraße 95, 14199 Berlin  13.308440  52.478876  
        446899194                                    ,  13.448016  52.468671

Since the full address has already been combined into the full_address column,  
the original address fields (addr:street, addr:housenumber, addr:postcode, addr:city) are no longer needed.  

Therefore, I remove them using drop().  
The parameter errors="ignore" ensures that the code does not fail in case any of these columns are missing.


In [12]:
petstores_cleaned.info()


<class 'geopandas.geodataframe.GeoDataFrame'>
MultiIndex: 96 entries, ('node', np.int64(111810614)) to ('way', np.int64(1195398657))
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   geometry       96 non-null     geometry
 1   name           96 non-null     object  
 2   brand          52 non-null     object  
 3   opening_hours  78 non-null     object  
 4   phone          20 non-null     object  
 5   website        30 non-null     object  
 6   full_address   96 non-null     object  
 7   longitude      96 non-null     float64 
 8   latitude       96 non-null     float64 
dtypes: float64(2), geometry(1), object(6)
memory usage: 10.0+ KB


This allows me to verify:
* Number of rows and columns  
* Which columns still contain missing values  
* Data types  

It’s a quick validation step to make sure the cleaning worked properly.


In [13]:
petstores_cleaned.head(20)


geometry                    name  \
element id                                                              
node    111810614   POINT (13.28841 52.49974)               Fressnapf   
        346138343   POINT (13.36752 52.54252)               Fressnapf   
        417360251   POINT (13.45196 52.53361)               Fressnapf   
        438385039   POINT (13.30844 52.47888)          Lucas Tierwelt   
        446899194   POINT (13.44802 52.46867)          Das Futterhaus   
        701825313   POINT (13.32671 52.49079)          Natürlich Hund   
        800755193     POINT (13.3535 52.5746)              Hundesalon   
        861870108   POINT (13.61083 52.54018)               Fressnapf   
        1112653257  POINT (13.24238 52.42322)               Fressnapf   
        1157588586   POINT (13.3666 52.56424)          Das Futterhaus   
        1198087358  POINT (13.19667 52.53016)               Fressnapf   
        1311479655  POINT (13.30647 52.52553)            Zoo fridolin   
        1427998890  POINT (13.12883 52.52843)               Fressnapf   
        1457600608  POINT (13.37072 52.56473)               Fressnapf   
        1498303230  POINT (13.55979 52.50635)          Das Futterhaus   
        1511328799   POINT (13.4244 52.48789)          Das Futterhaus   
        1608667951  POINT (13.38438 52.50312)               Fressnapf   
        1643105094  POINT (13.33338 52.59533)  Marine Aquarium Senior   
        1774557661  POINT (13.34222 52.46893)      Steffis Futterkeks   
        1779217862  POINT (13.50724 52.49925)               Fressnapf   

                             brand  \
element id                           
node    111810614        Fressnapf   
        346138343        Fressnapf   
        417360251        Fressnapf   
        438385039              NaN   
        446899194   Das Futterhaus   
        701825313              NaN   
        800755193              NaN   
        861870108        Fressnapf   
        1112653257       Fressnapf   
        1157588586  Das Futterhaus   
        1198087358       Fressnapf   
        1311479655             NaN   
        1427998890       Fressnapf   
        1457600608       Fressnapf   
        1498303230  Das Futterhaus   
        1511328799  Das Futterhaus   
        1608667951       Fressnapf   
        1643105094             NaN   
        1774557661             NaN   
        1779217862       Fressnapf   

                                                        opening_hours  \
element id                                                              
node    111810614                   Mo-Fr 09:00-20:00; Sa 09:00-19:00   
        346138343                   Mo-Fr 09:00-20:00; Sa 09:00-18:00   
        417360251                                                 NaN   
        438385039                                   Mo-Sa 08:00-20:00   
        446899194                                                 NaN   
        701825313                   Tu-Fr 11:00-18:00; Sa 10:00-15:00   
        800755193   "nach Vereinbarung (Beschriftung der Öffnungsz...   
        861870108           Mo-Fr 09:00-20:00; Sa 09:00-18:00; PH off   
        1112653257                                                NaN   
        1157588586                                             closed   
        1198087358                       Mo-Sa 09:00-20:00; Su,PH off   
        1311479655  Mo-Fr 09:00-18:00; Sa 09:00-14:00; Su off, PH off   
        1427998890                                                NaN   
        1457600608                  Mo-Fr 09:00-20:00; Sa 09:00-18:00   
        1498303230                                                NaN   
        1511328799                                  Mo-Sa 10:00-20:00   
        1608667951                  Mo-Fr 09:00-20:00; Sa 09:00-18:00   
        1643105094                                  Mo-Sa 10:00-18:00   
        1774557661  Mo-Th 09:00-12:00,14:00-18:00; Fr 09:00-12:00,...   
        1779217862                  Mo-Fr 09:00-20:00; Sa 09

Using petstores_cleaned.head(20) I preview the first 20 rows of the cleaned dataset.  

This helps me visually confirm that:  
* The selected columns are correct  
* The full_address column was created properly  
* Latitude and longitude values look valid  
* The data structure matches the expected format


In [14]:
print("Final dataset shape:", petstores_cleaned.shape)
print("Columns:", list(petstores_cleaned.columns))
print("Any missing values left?", petstores_cleaned.isna().any().sum())


Final dataset shape: (96, 9)
Columns: ['geometry', 'name', 'brand', 'opening_hours', 'phone', 'website', 'full_address', 'longitude', 'latitude']
Any missing values left? 4


In [15]:
petstores_cleaned.to_csv("berlin_pet_stores_cleaned.csv", index=False)


* This line allows me to export the cleaned dataset. So I am saving the petstores_cleanedtable as a CSV file on my computer .

* berlin_pet_stores_cleaned.csv -- This is the name of the file I am saving.  
* index=False → I am not including row index numbers in the CSV because they are not needed. 

Data Cleaning & Standardization Notes

Steps I Followed During Data Cleaning

* I first loaded the raw OSM data into a GeoDataFrame called petstores_gdf.

* I reviewed the full list of available columns to understand what information the dataset contained.

* I used isna().sum() to check the amount of missing data in each column.

* Columns that were mostly empty or not relevant to the project were removed.

* I selected a meaningful subset of columns and created a cleaned dataset: petstores_cleaned.

Standardizing Columns

* Address fields (addr:street, addr:housenumber, addr:postcode, addr:city) were combined into a single, readable full_address column.

* To work with geographic data more easily, I extracted latitude and longitude from the geometry column.

* The original, now redundant address columns were removed.


Data Sources and Integration

* At this stage, only OSM was used as the primary data source.

 Challenges I Faced

* The most challenging part for me was deciding which columns were actually useful.
The raw OSM dataset contains many fields, and the column names are not always intuitive.
So I needed time to understand what each field represented.

* Using isna().sum() to examine missing data helped me see which columns had meaningful values and which were mostly empty.
This made the column selection process much clearer.

* I also needed to pay attention when creating the full_address column because some rows were missing one or more address components.
To handle this, I used fillna(''), which allowed me to merge the address parts cleanly without errors.

* Overall, once I understood the structure of the data, the workflow became more natural and easier to follow.

Deciding Which Data to Keep

 I kept these columns because they provide direct value to the user and to the application:

* name — Important for identifying the store

* geometry — Contains the raw geographic location data

* brand — Needed for segmenting and categorizing stores by brand

* full_address — Provides a clean and readable address format

* opening_hours — Useful for user decision-making

* phone, website — Helps with direct communication

* latitude, longitude — Necessary for map display and spatial analysis

 Columns removed — because they were too incomplete or not relevant:

* Payment-related fields (payment:*)

* OSM metadata (source, wikidata)